In [ ]:
# -*- coding: utf-8 -*-
"""
Vérification par solution manufacturée (MMS) — système couplé h–C
Modèle vérifié :

  (1)  div( Kf grad h ) + div( Kf * buoy(C) * e_z ) = S_h
  (2)  div( q C ) - div( D(q) grad C )              = S_C   (régime permanent)

  q = -Kf ( grad h + buoy(C) e_z ),   buoy(C) = drho_dC * C / rho_f

Méthode : on choisit h_ex(x,z) et C_ex(x,z) analytiques, on calcule S_h et
S_C par calcul symbolique (sympy), on les injecte comme sources, et on
vérifie que la solution discrète converge vers (h_ex, C_ex) quand dx -> 0 :

  attendu : ordre 2 pour h (diffusion centrée)
            ordre 1 pour C (convection upwind)
            div(q) au zéro machine sur toutes les grilles

Le couplage h <-> C est résolu par itérations de Picard à chaque grille.
"""

import numpy as np
import sympy as sp
from fipy import (Grid2D, CellVariable, FaceVariable,
                  DiffusionTerm, UpwindConvectionTerm)
from fipy.solvers.scipy import LinearLUSolver


# Paramètres physiques

Kf, ne          = 1e-4, 0.25
rho_f, rho_s    = 1000.0, 1025.0
C_mer           = 35.0
drho_dC         = (rho_s - rho_f) / C_mer
alpha_L, alpha_T = 1.0, 0.1
Dm              = 1e-9
L, H            = 100.0, 50.0


# 1) Solutions manufacturées + sources par calcul symbolique
x, z = sp.symbols('x z', real=True)

h_ex = sp.Float(0.02) * sp.sin(sp.pi * x / L) * sp.sin(sp.pi * z / H) \
       - sp.Float(0.01) * z / H
C_ex = sp.Float(C_mer/2) * (1 + sp.Rational(4, 5)
                            * sp.sin(sp.pi * x / L)
                            * sp.cos(sp.pi * z / (2*H)))   # dans [3.5, 31.5]

buoy_ex = drho_dC * C_ex / rho_f
qx_ex = -Kf * sp.diff(h_ex, x)
qz_ex = -Kf * (sp.diff(h_ex, z) + buoy_ex)
qn_ex = sp.sqrt(qx_ex**2 + qz_ex**2)

# Tenseur de dispersion 
Dxx_ex = alpha_L*qx_ex**2/qn_ex + alpha_T*qz_ex**2/qn_ex + ne*Dm
Dzz_ex = alpha_L*qz_ex**2/qn_ex + alpha_T*qx_ex**2/qn_ex + ne*Dm
Dxz_ex = (alpha_L - alpha_T)*qx_ex*qz_ex/qn_ex

# Source de l'équation en h : S_h = div(Kf grad h_ex) + div(Kf buoy e_z)
S_h_ex = (sp.diff(Kf*sp.diff(h_ex, x), x)
          + sp.diff(Kf*sp.diff(h_ex, z), z)
          + sp.diff(Kf*buoy_ex, z))

# Source du transport permanent : S_C = div(q C) - div(D grad C)
Fx = qx_ex*C_ex - (Dxx_ex*sp.diff(C_ex, x) + Dxz_ex*sp.diff(C_ex, z))
Fz = qz_ex*C_ex - (Dxz_ex*sp.diff(C_ex, x) + Dzz_ex*sp.diff(C_ex, z))
S_C_ex = sp.diff(Fx, x) + sp.diff(Fz, z)

f = lambda e: sp.lambdify((x, z), e, 'numpy')
h_f, C_f       = f(h_ex), f(C_ex)
S_h_f, S_C_f   = f(S_h_ex), f(S_C_ex)
qx_f, qz_f     = f(qx_ex), f(qz_ex)


# 2) Résolution discrète sur une grille 
def solve_on_grid(N, n_picard=40, tol_picard=1e-12):
    Nx, Nz = N, N // 2
    dx = L / Nx
    mesh = Grid2D(dx=dx, dy=dx, nx=Nx, ny=Nz)
    solver = LinearLUSolver(tolerance=1e-13, iterations=2000)

    xc = np.array(mesh.cellCenters[0]); zc = np.array(mesh.cellCenters[1])
    xf = np.array(mesh.faceCenters[0]); zf = np.array(mesh.faceCenters[1])
    ext = np.array(mesh.exteriorFaces)
    nrm = np.array(mesh.faceNormals)
    nF  = mesh.numberOfFaces

    h = CellVariable(mesh=mesh, value=0.0)
    C = CellVariable(mesh=mesh, value=float(C_mer/2))
    # Dirichlet exacts sur TOUTE la frontière (arrays complets sur les faces)
    h.constrain(h_f(xf, zf), mesh.exteriorFaces)
    C.constrain(C_f(xf, zf), mesh.exteriorFaces)

    Kf_face = FaceVariable(mesh=mesh, value=Kf)
    Kf_vals = np.full(nF, Kf)
    rho_cell = CellVariable(mesh=mesh, value=rho_f)
    rho_face = rho_cell.arithmeticFaceValue

    b_var  = CellVariable(mesh=mesh, value=0.0)
    S_C    = CellVariable(mesh=mesh, value=S_C_f(xc, zc))
    S_h    = CellVariable(mesh=mesh, value=S_h_f(xc, zc))
    D_face = FaceVariable(mesh=mesh, rank=2)
    q_face = FaceVariable(mesh=mesh, rank=1)
    q_face.setValue(np.zeros((2, nF)))
    buoy_flux = FaceVariable(mesh=mesh, rank=1)

    eq_h = DiffusionTerm(coeff=Kf_face) == b_var
    eq_C = (UpwindConvectionTerm(coeff=q_face)
            - DiffusionTerm(coeff=D_face)) == S_C

    div_max = 0.0
    for k in range(n_picard):
        C_old_it = C.value.copy()

        # h
        rho_cell.setValue(rho_f + drho_dC * C.value)
        buoy = (np.array(rho_face) - rho_f) / rho_f
        bf = np.zeros((2, nF)); bf[1] = Kf_vals * buoy
        buoy_flux.setValue(bf)
        b_var.setValue(-np.array(buoy_flux.divergence) + S_h.value)
        eq_h.solve(var=h, solver=solver)

        # q
        h_fg = h.faceGrad.value
        qx = -Kf_vals * h_fg[0]
        qz = -Kf_vals * (h_fg[1] + buoy)
        q_face.setValue(np.array([qx, qz]))

        
        tmp = FaceVariable(mesh=mesh, rank=1, value=np.array([qx, qz]))
        div_max = max(div_max,
                      np.abs(np.array(tmp.divergence) + S_h.value).max())

        # D
        qn = np.sqrt(qx**2 + qz**2) + 1e-300
        Da = np.zeros((2, 2, nF))
        Da[0, 0] = alpha_L*qx**2/qn + alpha_T*qz**2/qn + ne*Dm
        Da[1, 1] = alpha_L*qz**2/qn + alpha_T*qx**2/qn + ne*Dm
        Da[0, 1] = Da[1, 0] = (alpha_L - alpha_T)*qx*qz/qn
        D_face.setValue(Da)

        # C
        eq_C.solve(var=C, solver=solver)

        if np.abs(C.value - C_old_it).max() < tol_picard * C_mer:
            break

    V = dx * dx
    err_h = np.sqrt(np.sum((h.value - h_f(xc, zc))**2) * V)
    err_C = np.sqrt(np.sum((C.value - C_f(xc, zc))**2) * V)
    return dx, err_h, err_C, div_max, k + 1

# ------------------------------------------------------------------
# 3) Étude de convergence
# ------------------------------------------------------------------
print(f"{'N':>5} {'dx [m]':>8} {'||h-h_ex||L2':>14} {'ordre':>6} "
      f"{'||C-C_ex||L2':>14} {'ordre':>6} {'div(q)max':>11} {'Picard':>7}")

res = []
for N in [16, 32, 64, 128,256]:
    dx, eh, eC, dv, np_it = solve_on_grid(N)
    o_h = np.log2(res[-1][1] / eh) if res else float('nan')
    o_C = np.log2(res[-1][2] / eC) if res else float('nan')
    res.append((dx, eh, eC))
    print(f"{N:>5} {dx:>8.3f} {eh:>14.4e} {o_h:>6.2f} "
          f"{eC:>14.4e} {o_C:>6.2f} {dv:>11.2e} {np_it:>7}")


# 3b) Test découplé : équation en h seule 

def solve_h_decoupled(N):
    Nx, Nz = N, N // 2
    dx = L / Nx
    mesh = Grid2D(dx=dx, dy=dx, nx=Nx, ny=Nz)
    solver = LinearLUSolver(tolerance=1e-13, iterations=2000)
    xc = np.array(mesh.cellCenters[0]); zc = np.array(mesh.cellCenters[1])
    xf = np.array(mesh.faceCenters[0]); zf = np.array(mesh.faceCenters[1])
    nF = mesh.numberOfFaces

    h = CellVariable(mesh=mesh, value=0.0)
    h.constrain(h_f(xf, zf), mesh.exteriorFaces)
    Kf_face = FaceVariable(mesh=mesh, value=Kf)
    Kf_vals = np.full(nF, Kf)

    buoy = drho_dC * C_f(xf, zf) / rho_f        # flottabilité EXACTE aux faces
    bf = np.zeros((2, nF)); bf[1] = Kf_vals * buoy
    buoy_flux = FaceVariable(mesh=mesh, rank=1, value=bf)
    b_var = CellVariable(
        mesh=mesh,
        value=-np.array(buoy_flux.divergence) + S_h_f(xc, zc))

    eq_h = DiffusionTerm(coeff=Kf_face) == b_var
    eq_h.solve(var=h, solver=solver)
    return dx, np.sqrt(np.sum((h.value - h_f(xc, zc))**2) * dx * dx)

print("\n--- h découplé (flottabilité exacte) : ordre propre du solveur de h ---")
print(f"{'N':>5} {'dx [m]':>8} {'||h-h_ex||L2':>14} {'ordre':>6}")
prev = None
for N in [16, 32, 64, 128, 256]:
    dx, eh = solve_h_decoupled(N)
    o = np.log2(prev / eh) if prev else float('nan')
    prev = eh
    print(f"{N:>5} {dx:>8.3f} {eh:>14.4e} {o:>6.2f}")

print("\nAttendu : ordre ~2 pour h découplé ; en couplé, h est plafonné à "
      "l'ordre du C upwind (~1) via la flottabilité.")

# 3c) Test découplé : C seul
def solve_C_decoupled(N):
    Nx, Nz = N, N // 2
    dx = L / Nx
    mesh = Grid2D(dx=dx, dy=dx, nx=Nx, ny=Nz)
    solver = LinearLUSolver(tolerance=1e-13, iterations=2000)
    xc = np.array(mesh.cellCenters[0]); zc = np.array(mesh.cellCenters[1])
    xf = np.array(mesh.faceCenters[0]); zf = np.array(mesh.faceCenters[1])
    nF = mesh.numberOfFaces

    C = CellVariable(mesh=mesh, value=float(C_mer/2))
    C.constrain(C_f(xf, zf), mesh.exteriorFaces)   # Dirichlet exacts
    S_C = CellVariable(mesh=mesh, value=S_C_f(xc, zc))

    # q
    qx = qx_f(xf, zf)
    qz = qz_f(xf, zf)
    qx = np.broadcast_to(np.asarray(qx, dtype=float), (nF,)).copy()
    qz = np.broadcast_to(np.asarray(qz, dtype=float), (nF,)).copy()
    q_face = FaceVariable(mesh=mesh, rank=1, value=np.array([qx, qz]))

    # D
    qn = np.sqrt(qx**2 + qz**2) + 1e-300
    Da = np.zeros((2, 2, nF))
    Da[0, 0] = alpha_L*qx**2/qn + alpha_T*qz**2/qn + ne*Dm
    Da[1, 1] = alpha_L*qz**2/qn + alpha_T*qx**2/qn + ne*Dm
    Da[0, 1] = Da[1, 0] = (alpha_L - alpha_T)*qx*qz/qn
    D_face = FaceVariable(mesh=mesh, rank=2, value=Da)

    # C
    eq_C = (UpwindConvectionTerm(coeff=q_face)
            - DiffusionTerm(coeff=D_face)) == S_C
    eq_C.solve(var=C, solver=solver)

    err_C = np.sqrt(np.sum((C.value - C_f(xc, zc))**2) * dx * dx)
    return dx, err_C

print("\n--- C découplé (champ de Darcy exact) : ordre propre du transport ---")
print(f"{'N':>5} {'dx [m]':>8} {'||C-C_ex||L2':>14} {'ordre':>6}")
prev = None
for N in [16, 32, 64, 128, 256]:
    dx, eC = solve_C_decoupled(N)
    o = np.log2(prev / eC) if prev else float('nan')
    prev = eC
    print(f"{N:>5} {dx:>8.3f} {eC:>14.4e} {o:>6.2f}")